# Agentic Legacy Integrator for Mainframe-to-ML Migration
## 1. Legacy Input Analysis & Preprocessing

**Observation:**
The input is a legacy mainframe JCL (Job Control Language) script used for DFSMS (Data Facility Storage Management Subsystem) configuration. It contains several steps (`STEP021` to `STEP065`) that define storage groups, data classes, and storage classes for a DB2 database.

**Cleaning Strategy:**
1.  **Noise Reduction:** Filter out empty lines, JCL comments starting with `//*`, and generic system DD statements (like `SYSPRINT`, `SYSUDUMP`).
2.  **Step Identification:** Use regex to identify executable steps (e.g., `//STEP... EXEC`).
3.  **Command Parsing:** Extract the core `ISPSTART CMD(...)` blocks where the actual business logic resides.
4.  **Rule Extraction:** Use regex to parse parameters (e.g., `HIGHTHRS(85)`) into a structured JSON format (Key-Value pairs).

**Ambiguities:**
* Many parameters have empty parentheses (e.g., `MIGSYSNM()`). We need to decide whether to map these to `null` or omit them in the modern schema.
* The connection between `SCDS('CPAC.DFSMS.SCDS')` and the defined classes implies a relational dependency that needs to be preserved.

In [1]:
import json
import re

def parse_jcl_to_json(file_path):
    """
    Reads a JCL file line-by-line and extracts business rules cleanly.
    """
    try:
        with open(file_path, 'r') as file:
            lines = file.readlines()
    except FileNotFoundError:
        return f"Error: Could not find the file at {file_path}"

    structured_data = {
        "metadata": {
            "source_type": "Mainframe JCL",
            "description": "DFSMS Storage Management Rules"
        },
        "steps": []
    }

    current_step = None
    in_cmd = False
    cmd_buffer = ""

    for line in lines:
        # Step 1: Clean the line
        clean_line = line.strip()
        
        clean_line = re.sub(r'\[.*?\]', '', clean_line).strip()    

        if not clean_line or clean_line.startswith('//*'):
            continue

        # Step 2: Detect the Step Name (e.g., //STEP031 EXEC ...)
        step_match = re.match(r'^//(STEP\d+)\s+EXEC', clean_line)
        if step_match:
            current_step = step_match.group(1)
            continue

        # Step 3: Detect the start of a Command block
        if "CMD(" in clean_line:
            in_cmd = True
            cmd_buffer = clean_line.split("CMD(")[1]
            continue

        # Step 4: Accumulate parameters until the command closes
        if in_cmd:
            cmd_buffer += " " + clean_line
            if clean_line == ")" or (clean_line.endswith(")") and "+" not in clean_line):
                
                cmd_buffer = cmd_buffer.replace('+', ' ')
                
                action_match = re.search(r'(DEFINE|DISPLAY|ALTER)', cmd_buffer)
                action = action_match.group(1) if action_match else "UNKNOWN"
                
                params = re.findall(r'([A-Z0-9]+)\((.*?)\)', cmd_buffer)
                rules = {k: (v.strip() if v.strip() else None) for k, v in params}
                
                structured_data["steps"].append({
                    "step_name": current_step or "UNKNOWN_STEP",
                    "action": action,
                    "extracted_rules": rules
                })
                
                in_cmd = False
                cmd_buffer = ""

    return structured_data

# --- Execution ---
file_path = '../data/raw/sms'
parsed_rules = parse_jcl_to_json(file_path)

if isinstance(parsed_rules, str):
    print(parsed_rules)
else:
    with open('../data/processed/structured_rules.json', 'w') as out_file:
        json.dump(parsed_rules, out_file, indent=4)
        
    print("Parsing Successful! Here is a preview of the extracted rules:\n")
    print(json.dumps(parsed_rules, indent=4)[:800] + "\n...\n}")

Parsing Successful! Here is a preview of the extracted rules:

{
    "metadata": {
        "source_type": "Mainframe JCL",
        "description": "DFSMS Storage Management Rules"
    },
    "steps": [
        {
            "step_name": "STEP031",
            "action": "DEFINE",
            "extracted_rules": {
                "SCDS": "'CPAC.DFSMS.SCDS'",
                "STORGRP": "SGIXXX",
                "DESCR": "STORAGE GROUP FOR DB2",
                "AUTOMIG": "N",
                "MIGSYSNM": null,
                "AUTOBKUP": "N",
                "BKUPSYS": null,
                "AUTODUMP": null,
                "DMPSYSNM": null,
                "OVRFLOW": null,
                "EXTSGNM": null,
                "DUMPCLAS": null,
                "HIGHTHRS": "85",
                "LOWTHRS": "30",
                "GUARBKFR": "NOLIMIT",
             
...
}


## 2. Business Rule Extraction & Mapping

### 2.1. Core Business Rules Extracted
Based on the JSON payload generated in the previous step, several critical business rules and system constraints were identified from the legacy DFSMS configuration:

* **Threshold Logic (Auto-scaling Triggers):** The legacy system defines strict capacity thresholds. `HIGHTHRS: 85` indicates an upper limit of 85% utilization before action is required, and `LOWTHRS: 30` indicates a lower bound of 30%.
* **Resource Constraints:** `VOLCNT: 255` restricts the maximum number of storage volumes that can be allocated to this data class.
* **Availability & SLA Requirements:** `ACCSBTY: C` (Continuous) enforces a high-availability requirement, meaning the database (DB2) requires zero-downtime access.
* **Data Lifecycle & Retention:** `GUARBKFR: NOLIMIT` implies a continuous or unlimited backup guarantee without a strict expiration frequency.
* **Storage Formatting:** `DSNMTYP: EXT` and `IFEXT: R` define the structural requirements for the datasets (Extended format, Required).

### 2.2. Rule-to-Schema Mapping Table

| Legacy Concept (DFSMS) | Extracted Value | Modern Equivalent (Cloud / API) | Target JSON/YAML Field | Rationale |
| :--- | :--- | :--- | :--- | :--- |
| `STORGRP` (Storage Group) | `SGIXXX` | Cloud Storage Pool / K8s StorageClass | `storagePoolId` | Groups similar storage resources. Maps perfectly to a Kubernetes StorageClass. |
| `HIGHTHRS` / `LOWTHRS` | `85` / `30` | Auto-scaling Utilization Targets | `autoScaling.targetUtilization` | In modern cloud, static thresholds are replaced by dynamic auto-scaling policies based on percentage utilization. |
| `VOLCNT` | `255` | Max Resource Quota | `resourceQuotas.maxVolumes` | Hard limits in legacy systems translate to tenant or namespace resource quotas in modern platforms. |
| `ACCSBTY` | `C` (Continuous) | High Availability (Multi-AZ) | `sla.highAvailability: true` | Continuous access in mainframe translates to Multi-AZ (Availability Zone) deployment in AWS/Azure. |
| `AUTOMIG` | `N` (No) | Auto-Tiering Policy | `lifecyclePolicy.autoTiering: false` | Legacy manual migration indicates that dynamic storage tiering (e.g., S3 Standard to Glacier) should be disabled. |

### 2.3. Assumptions and Ambiguities
* **Assumption 1:** The legacy parameter `ACCSBTY: C` is assumed to require a Multi-Region or Multi-AZ architecture in the target cloud environment to guarantee continuous access.
* **Assumption 2:** Many parameters (e.g., `MIGSYSNM`, `AUTODUMP`) were extracted as `null`. We assume these were either default behaviors or non-essential features in the legacy system. In the modern schema, we will omit them to maintain a clean API contract.
* **Ambiguity:** The exact metric for `HIGHTHRS: 85` (whether it's IOPS, throughput, or purely disk capacity) is implied to be capacity based on typical DFSMS setups, but would require business stakeholder validation in a real-world scenario.

In [3]:
import json

def map_legacy_to_modern_schema(legacy_json):
    """
    Transforms the extracted legacy rules into a modern REST API payload
    suitable for a Cloud Storage Provisioning Microservice.
    """
    
    # We will extract specific rules from the legacy JSON to build our modern schema
    storage_group_rules = {}
    data_class_rules = {}
    storage_class_rules = {}

    for step in legacy_json.get("steps", []):
        if step.get("action") == "DISPLAY":
            continue
        rules = step.get("extracted_rules", {})
        if "STORGRP" in rules:
            storage_group_rules = rules
        if "DCNAME" in rules:
            data_class_rules = rules
        if "STCNAME" in rules:
            storage_class_rules = rules

    # Building the modern JSON schema (API Payload)
    modern_payload = {
        "apiVersion": "v1",
        "kind": "StorageProvisioningRequest",
        "metadata": {
            "serviceName": "DB2_Modernization_DB",
            "environment": "production"
        },
        "spec": {
            "storageClass": {
                "name": storage_class_rules.get("STCNAME", "default-class"),
                "description": storage_class_rules.get("DESCR", ""),
                "sla": {
                    # 'C' means Continuous, which we map to High Availability
                    "highAvailability": True if storage_class_rules.get("ACCSBTY") == "C" else False
                }
            },
            "capacityPlanning": {
                "quotas": {
                    "maxVolumes": int(data_class_rules.get("VOLCNT", 100))
                },
                "autoScaling": {
                    "enabled": True,
                    "scaleOutThresholdPercent": int(storage_group_rules.get("HIGHTHRS", 80)),
                    "scaleInThresholdPercent": int(storage_group_rules.get("LOWTHRS", 20))
                }
            },
            "dataLifecycle": {
                "autoTieringEnabled": True if storage_group_rules.get("AUTOMIG") == "Y" else False,
                "backupPolicy": storage_group_rules.get("GUARBKFR", "STANDARD")
            }
        }
    }
    
    return modern_payload

# --- Execution ---
# Load the processed legacy rules
with open('../data/processed/structured_rules.json', 'r') as f:
    legacy_data = json.load(f)

# Generate the modern schema
modernized_schema = map_legacy_to_modern_schema(legacy_data)

# Save the modern schema
with open('../data/processed/modern_api_payload.json', 'w') as f:
    json.dump(modernized_schema, f, indent=4)

print("Modernization Mapping Complete! Here is the target API schema:\n")
print(json.dumps(modernized_schema, indent=4))

Modernization Mapping Complete! Here is the target API schema:

{
    "apiVersion": "v1",
    "kind": "StorageProvisioningRequest",
    "metadata": {
        "serviceName": "DB2_Modernization_DB",
        "environment": "production"
    },
    "spec": {
        "storageClass": {
            "name": "SCIXXX",
            "description": "STORAGE CLASS FOR DB2",
            "sla": {
                "highAvailability": true
            }
        },
        "capacityPlanning": {
            "quotas": {
                "maxVolumes": 255
            },
            "autoScaling": {
                "enabled": true,
                "scaleOutThresholdPercent": 85,
                "scaleInThresholdPercent": 30
            }
        },
        "dataLifecycle": {
            "autoTieringEnabled": false,
            "backupPolicy": "NOLIMIT"
        }
    }
}
